In [1]:
!pip install -q crewai crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48

In [2]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [3]:
from crewai import LLM

llm = LLM(
    model = "gemini/gemini-flash-lite-latest",
    temperature = 0.2
)

In [4]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)

regions = ["Hyderabad", "Bangalore", "Chennai", "Pune"]
products = {"Laptop": 55000, "Phone": 20000, "Smartwatch": 8000, "Headphones": 2500}
dates = pd.date_range("2026-10-01", "2026-10-14")

rows = []
for d in dates:
    for region in regions:
        for product, price in products.items():
            units = int(rng.integers(1, 10))
            if product == "Smartwatch" and d >= pd.Timestamp("2026-10-08"):
                units = int(rng.integers(0, 3))
            if region == "Hyderabad":
                units += 2
            rows.append({"date": d.date(), "region": region, "product": product,
                         "units": units, "price": price})

df = pd.DataFrame(rows)
df.to_csv("sales.csv", index=False)

print(df.shape)
df.head()

(224, 5)


,date,region,product,units,price
0,2026-10-01,Hyderabad,Laptop,3,55000
1,2026-10-01,Hyderabad,Phone,9,20000
2,2026-10-01,Hyderabad,Smartwatch,8,8000
3,2026-10-01,Hyderabad,Headphones,6,2500
4,2026-10-01,Bangalore,Laptop,4,55000


In [5]:
df["revenue"] = df["units"] * df["price"]

print("Total revenue:", df["revenue"].sum())
print("\nRevenue by region:")
print(df.groupby("region")["revenue"].sum().sort_values(ascending=False))
print("\nRevenue by product:")
print(df.groupby("product")["revenue"].sum().sort_values(ascending=False))

df["week"] = np.where(pd.to_datetime(df["date"]) < "2026-10-08", "week1", "week2")
print("\nSmartwatch units by week:")
print(df[df["product"] == "Smartwatch"].groupby("week")["units"].sum())

Total revenue: 24831500

Revenue by region:
region
Hyderabad    8134500
Pune         6140000
Chennai      5694000
Bangalore    4863000
Name: revenue, dtype: int64

Revenue by product:
product
Laptop        16005000
Phone          6400000
Smartwatch     1664000
Headphones      762500
Name: revenue, dtype: int64

Smartwatch units by week:
week
week1    169
week2     39
Name: units, dtype: int64


In [6]:
import json
from crewai.tools import tool

@tool("Sales Stats Calculator")
def sales_stats_tool(file_path: str = "sales.csv") -> str:
    """Calculates exact sales statistics from a sales CSV file.
    Returns total revenue, revenue by region, revenue by product,
    and week-over-week unit changes per product. Use this instead of
    doing any math yourself."""
    try:
        data = pd.read_csv(file_path)
        data["revenue"] = data["units"] * data["price"]
        data["week"] = np.where(pd.to_datetime(data["date"]) < "2026-10-08",
                                "week1", "week2")

        units_by_week = data.groupby(["product", "week"])["units"].sum().unstack()
        units_by_week["change_pct"] = (
            (units_by_week["week2"] - units_by_week["week1"])
            / units_by_week["week1"] * 100
        ).round(1)

        stats = {
            "total_revenue": int(data["revenue"].sum()),
            "revenue_by_region": data.groupby("region")["revenue"].sum()
                                     .sort_values(ascending=False).to_dict(),
            "revenue_by_product": data.groupby("product")["revenue"].sum()
                                      .sort_values(ascending=False).to_dict(),
            "units_week1_vs_week2": units_by_week.to_dict(orient="index"),
        }
        return json.dumps(stats, indent=2, default=int)
    except Exception as e:
        return f"Could not calculate stats: {e}"

print(sales_stats_tool.run(file_path="sales.csv"))

{
  "total_revenue": 24831500,
  "revenue_by_region": {
    "Hyderabad": 8134500,
    "Pune": 6140000,
    "Chennai": 5694000,
    "Bangalore": 4863000
  },
  "revenue_by_product": {
    "Laptop": 16005000,
    "Phone": 6400000,
    "Smartwatch": 1664000,
    "Headphones": 762500
  },
  "units_week1_vs_week2": {
    "Headphones": {
      "week1": 163,
      "week2": 142,
      "change_pct": -12.9
    },
    "Laptop": {
      "week1": 155,
      "week2": 136,
      "change_pct": -12.3
    },
    "Phone": {
      "week1": 164,
      "week2": 156,
      "change_pct": -4.9
    },
    "Smartwatch": {
      "week1": 169,
      "week2": 39,
      "change_pct": -76.9
    }
  }
}


In [7]:
##Defining the Structed output using pydantic
from pydantic import BaseModel, Field

class SalesStats(BaseModel):
    total_revenue: int = Field(description="Total revenue across all sales")
    top_region: str = Field(description="Region with the highest revenue")
    top_region_revenue: int = Field(description="Revenue of the top region")
    top_product: str = Field(description="Product with the highest revenue")
    top_product_revenue: int = Field(description="Revenue of the top product")
    biggest_drop_product: str = Field(description="Product with the largest week-over-week fall in units")
    biggest_drop_pct: float = Field(description="Percentage change in units for that product, e.g. -76.9")

In [8]:
from crewai import Agent

analyst = Agent(
    role="Data Analyst",
    goal="Get exact sales statistics using the Sales Stats Calculator tool",
    backstory="You never calculate numbers yourself. You always use the "
              "Sales Stats Calculator tool and copy its numbers exactly.",
    tools=[sales_stats_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

writer = Agent(
    role="Insight Writer",
    goal="Turn sales statistics into the top 3 findings a manager should know",
    backstory="You explain data in simple business language. You only use "
              "numbers given to you and never invent or recalculate them.",
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

reviewer = Agent(
    role="Report Reviewer",
    goal="Make sure every number in the report exactly matches the real statistics",
    backstory="You are a strict fact-checker. You re-run the Sales Stats "
              "Calculator yourself and compare every number in the report "
              "against it. You fix any number that does not match.",
    tools=[sales_stats_tool],
    llm=llm,
    verbose=True,
    allow_delegation=False,
)


In [9]:
from crewai import Task

analysis_task = Task(
    description="Use the Sales Stats Calculator tool on 'sales.csv'. From its "
                "output, fill in the total revenue, the top region and its "
                "revenue, the top product and its revenue, and the product with "
                "the biggest week-over-week drop in units with its change_pct. "
                "Copy the numbers exactly from the tool.",
    expected_output="The sales statistics in the required structured format.",
    agent=analyst,
    output_pydantic=SalesStats,
)

insight_task = Task(
    description="Using the sales statistics, write a short weekly sales report "
                "for a manager. Start with one line giving the total revenue, "
                "then give the top 3 findings, each with the exact number that "
                "supports it and one suggested action.",
    expected_output="A short Markdown report with a total revenue line and "
                    "3 numbered findings, each with a number and an action.",
    agent=writer,
    context=[analysis_task],
)

review_task = Task(
    description="Review the sales report. Run the Sales Stats Calculator tool "
                "yourself and check every number in the report against it. "
                "Fix any number that is wrong. Return the final corrected "
                "report only, with no comments about your review.",
    expected_output="The final, fact-checked sales report in Markdown.",
    agent=reviewer,
    context=[analysis_task, insight_task],
    output_file="report.md",
)

In [10]:
from crewai import Crew, Process

report_crew = Crew(
    agents = [analyst, writer, reviewer],
    tasks = [analysis_task, insight_task, review_task],
    process = Process.sequential,
    max_rpm = 8,
    verbose = True,
)
result = await report_crew.kickoff_async()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: bae5fd39-b015-4892-ad7a-7f35268a7205                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the Sales Stats Calculator tool on 'sales.csv'. From its output, fill in the total revenue, the top  │
│  region and its revenue, the top product and its revenue, and the product with the biggest week-over-week drop  │
│  in units with its change_pct. Copy the numbers exactly from the tool.                                          │
│  ID: e2470867-5f1b-4535-afd7-7febdfa552eb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Data Analyst                                                                                            │
│                                                                                                                 │
│  Task: Use the Sales Stats Calculator tool on 'sales.csv'. From its output, fill in the total revenue, the top  │
│  region and its revenue, the top product and its revenue, and the product with the biggest week-over-week drop  │
│  in units with its change_pct. Copy the numbers exactly from the tool.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: sales_stats_calculator                                                                                   │
│  Args: {"file_path": "sales.csv"}                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "total_revenue": 24831500,                                                                                   │
│    "revenue_by_region": {                                                                                       │
│      "Hyderabad": 8134500,                                                                                      │
│      "Pune": 6140000,                                                                                           │
│      "Chennai": 5694000,                                                                                        │
│      "Bangalore": 4863000                                                                                       │
│    },                                                                                                           │
│    "revenue_by_product": {                                                                                      │
│      "Laptop": 16005000,                                                                                        │
│      "Phone": 6400000,                                                                                          │
│      "Smartwatch": 1664000,                                                                                     │
│      "Headphones": 762500                                                                                       │
│    },                                                                                                           │
│    "units_week1_vs_week2": {                                                                                    │
│      "Headphones": {                                                                                            │
│        "week1": 163,                                                                                            │
│        "week2": 142,                                                                                            │
│        "change_pct": -12.9                                                                                      │
│      },                                                                                                         │
│      "Laptop": {                                                                                                │
│        "week1": 155,                                                                                            │
│        "week2": 136,                                                                                            │
│        "change_pct": -12.3                                                                                      │
│      },                                                                                                         │
│      "Phone": {                                                                                                 │
│        "week1": 164,                                                                                            │
│        "week2": 156,                                                                                            │
│        "change_pct": -4.9                                                                                       │
│      },                                                                                                         │
│      "Smartwatch": {                                                                                            │
│        "week1": 169,                                                                                            │
│        "week2": 39,                                   

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#2) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: sales_stats_calculator                                                                                   │
│  Output: {                                                                                                      │
│    "total_revenue": 24831500,                                                                                   │
│    "revenue_by_region": {                                                                                       │
│      "Hyderabad": 8134500,                                                                                      │
│      "Pune": 6140000,                                                                                           │
│      "Chennai": 5694000,                                                                                        │
│      "Bangalore": 4863000                                                                                       │
│    },                                                                                                           │
│    "revenue_by_product": {                                                                                      │
│      "Laptop": 16005000,                                                                                        │
│      "Phone": 6400000,                                                                                          │
│      "Smartwatch": 1664000,                                                                                     │
│      "Headphones": 762500                                                                                       │
│    },                                                                                                           │
│    "units_week1_vs_week2": {                                                                                    │
│      "Headphones": {                                                                                            │
│        "week1": 163,                                                                                            │
│        "week2": 142,                                                                                            │
│        "change_pct": -12.9                                                                                      │
│      },                                                                                                         │
│      "Laptop": {                                                                                                │
│        "week1": 155,                                                                                            │
│        "week2": 136,                                                                                            │
│        "change_pct": -12.3                                                                                      │
│      },                                                                                                         │
│      "Phone": {                                                                                                 │
│        "week1": 164,                                                                                            │
│        "week2": 156,                                                                                            │
│        "change_pct": -4.9                                                                                       │
│      },                                                                                                         │
│      "Smartwatch": {                                   

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Report Reviewer                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Total Revenue: 24,831,500                                                                                      │
│                                                                                                                 │
│  1. **Top Performing Region:** Hyderabad generated the highest revenue, bringing in 8,134,500.                  │
│     - *Suggested Action:* Allocate more marketing resources to Hyderabad to sustain and expand this strong      │
│  regional momentum.                                                                                             │
│                                                                                                                 │
│  2. **Best-Selling Product:** Laptops drove the most sales, contributing 16,005,000 to the total revenue.       │
│     - *Suggested Action:* Ensure sufficient laptop inventory is maintained to prevent stockouts and meet high   │
│  customer demand.                                                                                               │
│                                                                                                                 │
│  3. **Significant Product Decline:** Smartwatches experienced the biggest drop in sales, falling by -76.9%.     │
│     - *Suggested Action:* Review pricing strategies or promotional campaigns for smartwatches to identify the   │
│  cause of the sharp decline and reverse the trend.                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the sales report. Run the Sales Stats Calculator tool yourself and check every number in the      │
│  report against it. Fix any number that is wrong. Return the final corrected report only, with no comments      │
│  about your review.                                                                                             │
│  Agent: Report Reviewer                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: bae5fd39-b015-4892-ad7a-7f35268a7205                                                                       │
│  Final Output: Total Revenue: 24,831,500                                                                        │
│                                                                                                                 │
│  1. **Top Performing Region:** Hyderabad generated the highest revenue, bringing in 8,134,500.                  │
│     - *Suggested Action:* Allocate more marketing resources to Hyderabad to sustain and expand this strong      │
│  regional momentum.                                                                                             │
│                                                                                                                 │
│  2. **Best-Selling Product:** Laptops drove the most sales, contributing 16,005,000 to the total revenue.       │
│     - *Suggested Action:* Ensure sufficient laptop inventory is maintained to prevent stockouts and meet high   │
│  customer demand.                                                                                               │
│                                                                                                                 │
│  3. **Significant Product Decline:** Smartwatches experienced the biggest drop in sales, falling by -76.9%.     │
│     - *Suggested Action:* Review pricing strategies or promotional campaigns for smartwatches to identify the   │
│  cause of the sharp decline and reverse the trend.                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯